In [39]:
import pandas as pd
from thefuzz import process
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# For imputation
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, clone
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from typing import Dict

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor


In [40]:
train_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/train.csv")

In [41]:
train_df.set_index("carID")

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,,
69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37194,Mercedes,C Class,2015.0,13498,Manual,14480.0,etrol,125.0,53.300000,2.0,78.0,0.000000,0.0
6265,Audi,Q3,2013.0,12495,Semi-Auto,52134.0,Diesel,200.0,47.900000,2.0,38.0,2.000000,0.0
54886,Toyota,Aygo,2017.0,8399,Automatic,11304.0,Petrol,145.0,67.000000,1.0,57.0,3.000000,0.0


# Data cleaning

In [42]:
# String cleaning and Small numbers changes

def simple_processing(df):
    """
    Apply string cleaning, brand/model corrections, and fuzzy matching.
    These operations don"t require fitting on training data.
    """

    df = df.copy()
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
        
    # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
    df["Brand"] = df["Brand"].str.lower().str.strip()
    df["model"] = df["model"].str.lower().str.strip()
    df["transmission"] = df["transmission"].str.lower().str.strip()
    df["fuelType"] = df["fuelType"].str.lower().str.strip()
    
    # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
    df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
    
    # 1.1 Fixing brands
    df["Brand"] = df["Brand"].map(brand_mapping)
    
    # 1.2 Fixing models
    # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
    # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
    
    # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
    
    # Models - handle different lengths separately
    unique_models = df["model"].unique()
    model_lookup = {}
    for val in unique_models:
        if pd.isna(val) or val == "NaN":
            model_lookup[val] = "NaN"
        elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
            model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
        elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
            model_lookup[val] = process.extractOne(val, short_models)[0]
        else:  # We can define models with only one letter
            model_lookup[val] = "NaN"
    df["model"] = df["model"].map(model_lookup)
    
    # Transmission
    unique_trans = df["transmission"].unique()
    trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
    df["transmission"] = df["transmission"].map(trans_lookup)
    
    # FuelType
    unique_fuel = df["fuelType"].unique()
    fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
    df["fuelType"] = df["fuelType"].map(fuel_lookup)
    
    # Convert the str NaN values back to pd.NA for easier further processing and readability
    df["model"] = df["model"].replace("NaN", pd.NA)
    df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
    df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
    
    # Get the most frequent brand for each model -> returns df with model and brand
    brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
    df = pd.merge(df, brand_models, on="model", how="left", suffixes=("", "_mode"))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
    
    df["Brand"] = df["Brand"].fillna(df["Brand_mode"])  # rename new column
    df.drop("Brand_mode", axis=1, inplace=True)  # remove the old brand column
    
    ################################################################################
    # Simple Number Cleaning
    ################################################################################

    # Cleaning numeric columns
    df["year"] = df["year"].round(0)
    
    # Create the new column
    df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
    df = df.drop(["hasDamage"], axis=1)

    # Round year to integer (no fractional years)
    df["year"] = df["year"].round()
    
    # Mileage: take absolute value and round
    # Some imputation might produce small negative values
    df["mileage"] = abs(df["mileage"].round())
    
    # Tax: take absolute value and round
    df["tax"] = abs(df["tax"].round())
    
    # MPG: round to 1 decimal place 
    df["mpg"] = abs(df["mpg"].round(1))
    
    # Engine size: round to 1 decimal place
    df["engineSize"] = abs(df["engineSize"].round(1))
    
    # Paint quality correction (domain-specific business logic)
    # Assumption based on data exploration:
    # - Values < 4 likely had decimal point in wrong place (e.g., 3.5 -> 35%)
    # - Values > 100 likely have erroneous leading 1 (e.g., 185 -> 85%)
    def fix_paint_quality(x):
        if x < 4:
            return x * 10
        elif x > 100:
            return x - 100
        else:
            return x
    
    df["paintQuality%"] = df["paintQuality%"].apply(fix_paint_quality).round()
    
    # Previous owners: take absolute value and round to integer
    df["previousOwners"] = abs(df["previousOwners"].round())
    
    return df

In [43]:
# Categorical featue Encoding
def fit_transform_encoding(df):
    """
    Fit label encoders for categorical columns on training data.
    """
    
    encoders = {
        "Brand": LabelEncoder(),
        "model": LabelEncoder(),
        "transmission": LabelEncoder(),
        "fuelType": LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    """encoders["brand"].fit(df["Brand"])
    encoders["model"].fit(df["model"])
    encoders["transmission"].fit(df["transmission"])
    encoders["fuelType"].fit(df["fuelType"])"""

    columns = ["Brand", "model", "transmission", "fuelType"]

    # Code adapted from: https://stackoverflow.com/questions/36808434/label-encoder-encoding-missing-values

    for column in columns:
        # Get non-null string values
        mask = df[column].notna() & (df[column].apply(type) == str)
        
        # Fit encoder on unique non-null values
        fit_by = df.loc[mask, column].unique()
        encoders[column].fit(fit_by)
        
        # Transform only non-null values (vectorized)
        new_col_name = column + "_transformed"
        df[new_col_name] = pd.NA  # Initialize with NA
        df.loc[mask, new_col_name] = encoders[column].transform(df.loc[mask, column])
        
        # Convert to nullable integer
        df[new_col_name] = df[new_col_name].astype("Int64")

    df = df.drop(columns, axis=1)
    return df, encoders

In [44]:
# Train imputer on train

def fit_imputer(df, fast=True): 
    # Select estimator based on speed/accuracy tradeoff
    if fast:
        # Use default BayesianRidge (fast, ~1 second)
        estimator = None
    else:
        # Use Random Forest for better accuracy with complex relationships (~2 minutes)
        estimator = RandomForestRegressor(
            n_estimators=20,      # Limited trees for speed
            max_depth=10,         # Prevent overfitting
            random_state=12       # Reproducibility
        )

    # TODO: for later
    # Test if we perform better if we use numerical and categorical imputers separately
    
    # Initialize imputer
    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="mean"         # Initial fill before iterative process
    )
    
    # TODO: Test other categorical imputers like: missForest, datawig
    """imputer = IterativeImputer(
        estimator=RandomForestClassifier(),
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="most_frequent"         # Initial fill before iterative process
        )"""

    # FIT on training data
    # CRITICAL: We fit on data that still has missing values!
    # The imputer learns patterns of missingness and relationships

    imputer.fit(df)
    
    return imputer

In [45]:
def apply_imputer(df, imputer):
    """
        Apply the pretrained imputer to the dataframe
    """

    imputed_values = imputer.transform(df)

    df[df.columns] = imputed_values

    # TODO: rounding is not the best approach as the imputers prediction are continues thus 1.2 doesnt mean the value is closer to 1 than 2
    # However, rounding is the quickest way to fix this for now
    df[["mpg", "engineSize"]] = abs(df[["mpg", "engineSize"]]).round(1)
    try: 
        df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "price", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    except KeyError: # will raise keyError if we run it on testing data as it has no price column
        df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]] = abs(df[["year", "tax", "mileage", "paintQuality%", "previousOwners", "Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"]]).round(0).astype(int)
    
    return df

In [46]:
def decode(df, encoders):
    # Iterative imputer produces ~20-30 values that are outside of the range of the encoder
    # The simplest fix is to clip does values back into the range of the encoder
 

    df["Brand_transformed"] = df["Brand_transformed"].clip(lower=0, upper=encoders["Brand"].classes_.shape[0]-1).astype(int)
    df["Brand"] = encoders["Brand"].inverse_transform(df["Brand_transformed"])

    df["transmission_transformed"] = df["transmission_transformed"].clip(lower=0, upper=encoders["transmission"].classes_.shape[0]-1).astype(int)
    df["transmission"] = encoders["transmission"].inverse_transform(df["transmission_transformed"])
        
    # Use clip with the information of the fitted encoder, .classes_.shape gives us the dimension of the labels the encoder uses [0] is the rows - 1 because we start clipping at 0
    df["model_transformed"] = df["model_transformed"].clip(lower=0, upper=encoders["model"].classes_.shape[0]-1).astype(int)
    df["model"] = encoders["model"].inverse_transform(df["model_transformed"])
    
    df["fuelType_transformed"] = df["fuelType_transformed"].clip(lower=0, upper=encoders["fuelType"].classes_.shape[0]-1).astype(int)
    df["fuelType"] = encoders["fuelType"].inverse_transform(df["fuelType_transformed"])
    

    df.drop(columns=["Brand_transformed", "model_transformed", "transmission_transformed", "fuelType_transformed"], inplace=True)

    return df

## Workflow for Train and Validation Sets

In [47]:
# Fix typos and small numeric cleanup
df = simple_processing(train_df)
# Encode cateogrical columns and replace the str with int columns. Return fitted encoders for decoding at the end
df_encoded, encoders = fit_transform_encoding(df)

In [64]:
stratify_col = df_encoded["Brand_transformed"].fillna(-999) # Fill the ~40 missing values with -999
# Train validation split
train_split, validation_split = train_test_split(df_encoded, test_size=0.2, random_state=42, stratify=stratify_col)


In [65]:
share_train = train_split["Brand_transformed"].value_counts() / train_split.shape[0]
share_validation= validation_split["Brand_transformed"].value_counts() / validation_split.shape[0]

In [53]:

# Train imputer on train (data leakage risk thus only train on train_split)
imputer = fit_imputer(train_split)

# Apply trained imputer to both datasplits
imputed_train = apply_imputer(train_split, imputer)
imputer_test = apply_imputer(validation_split, imputer)

# Decode encoded columns using the fitted encoders
train_processed = decode(imputed_train, encoders)
validation_processed = decode(imputer_test, encoders)

## Workflow for Seperated Testing Dataset

In [54]:
test_df = pd.read_csv("https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/refs/heads/main/data/test.csv")

In [55]:
test_df.set_index("carID")

,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
carID,,,,,,,,,,,,
89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
100174,Opel,Grandland X,2019.000000,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
81376,BMW,1 Series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
105775,VW,Tiguan,2017.000000,Manual,27575.000000,Petrol,145.0,46.3,1.4,94.0,1.0,0.0
81363,BMW,X2,2020.000000,Automatic,1980.000000,Petrol,145.0,34.0,2.0,39.0,3.0,0.0
76833,Audi,Q5,2019.000000,Semi-Auto,8297.000000,Diesel,145.0,38.2,2.0,88.0,4.0,0.0


In [56]:
test = simple_processing(test_df)
test_df_encoded, test_encoders = fit_transform_encoding(test)


df_encoded_no_price = df_encoded.drop(["price"], axis=1)
test_imputers = fit_imputer(df_encoded_no_price) # we use the full encoded training dataframe to train the imputers

# Apply trained imputer to both datasplits
test_df_imputed = apply_imputer(test_df_encoded, test_imputers)

test_processed = decode(test_df_imputed, test_encoders)


# Test of the models on the full dataset

In [57]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

class BrandModelTrainer:
    def __init__(self, estimator):
        self.estimator = estimator
        self.brand_models = {}
        self.feature_cols = None

    def fit(self, X_train, y_train):
        self.feature_cols = [c for c in X_train.columns if c != "Brand"]
        print(f"Training models for {len(X_train['Brand'].unique())} brands...\n")

        for brand in X_train["Brand"].unique():
            mask = X_train["Brand"] == brand
            Xb = X_train.loc[mask, self.feature_cols]
            yb = y_train[mask]

            numeric_cols = Xb.select_dtypes(include=["int64", "float64"]).columns.tolist()
            categorical_cols = Xb.select_dtypes(include=["object", "category"]).columns.tolist()

            preprocessor = ColumnTransformer([
                ("num", RobustScaler(), numeric_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
            ])

            model = Pipeline([
                ("preprocess", preprocessor),
                ("estimator", clone(self.estimator))
            ])

            model.fit(Xb, yb)
            self.brand_models[brand] = model
            print(f"  ✓ {brand} done.")

        return self

    def predict(self, X):
        preds = np.zeros(len(X))
        for brand, model in self.brand_models.items():
            mask = X["Brand"] == brand
            if mask.sum() == 0:
                continue
            Xb = X.loc[mask, self.feature_cols]
            preds[mask] = model.predict(Xb)
        return preds

    # --- metriche overall ---
    def evaluate_train(self, X_train, y_train):
        y_pred = self.predict(X_train)
        rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        mae = mean_absolute_error(y_train, y_pred)
        r2 = r2_score(y_train, y_pred)
        print("\nTraining Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    def evaluate(self, X_val, y_val):
        y_pred = self.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        print("\nValidation Set Performance (Overall):")
        print(f"  RMSE: {rmse:.2f}")
        print(f"  MAE:  {mae:.2f}")
        print(f"  R²:   {r2:.4f}")
        return {"RMSE": rmse, "MAE": mae, "R²": r2}

    # --- metriche per brand ---
    def evaluate_by_brand(self, X, y, split_name="Validation"):
        y_pred = self.predict(X)
        results = []
        for brand in X["Brand"].unique():
            mask = X["Brand"] == brand
            y_true_b = y[mask]
            y_pred_b = y_pred[mask]
            rmse = np.sqrt(mean_squared_error(y_true_b, y_pred_b))
            mae = mean_absolute_error(y_true_b, y_pred_b)
            r2 = r2_score(y_true_b, y_pred_b)
            results.append({"Brand": brand, "N": len(y_true_b), "RMSE": rmse, "MAE": mae, "R²": r2})
        df = pd.DataFrame(results).sort_values("RMSE")
        print(f"\n{split_name} Performance per Brand:")
        print(df.to_string(index=False))
        return df

    def evaluate_train_by_brand(self, X_train, y_train):
        return self.evaluate_by_brand(X_train, y_train, split_name="Training")


In [58]:
X_train = train_processed.drop(columns=['price'])
y_train = train_processed['price']
X_val = validation_processed.drop(columns=['price'])
y_val = validation_processed['price']

### Linear Regression

In [59]:
# istanza del modello
LR = LinearRegression()
trainer = BrandModelTrainer(LR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ toyota done.
  ✓ audi done.
  ✓ skoda done.
  ✓ opel done.
  ✓ vw done.
  ✓ bmw done.
  ✓ ford done.
  ✓ hyundai done.

Training Set Performance (Overall):
  RMSE: 3572.08
  MAE:  2149.96
  R²:   0.8651

Validation Set Performance (Overall):
  RMSE: 3599.91
  MAE:  2199.98
  R²:   0.8644

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7635 1569.254729 1092.444043 0.804812
  toyota  3774 1871.556076 1171.804225 0.909472
    ford 13114 2160.033966 1537.315802 0.797238
 hyundai  2723 2202.855432 1564.511104 0.860865
   skoda  3510 2433.683000 1543.361495 0.848094
      vw  8485 2945.533988 2073.737045 0.853896
    audi  5973 4544.218407 2915.449391 0.848650
     bmw  6037 4568.821648 3040.240855 0.840769
mercedes  9527 5843.617510 3642.866951 0.727802

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1909 1647.476974 1142.267382 0.791766
   skoda  876

,Brand,N,RMSE,MAE,R²
6,opel,1909,1647.476974,1142.267382,0.791766
0,skoda,876,2167.738101,1532.707068,0.878963
5,ford,3278,2184.559128,1558.582527,0.787289
4,toyota,944,2245.417319,1249.254934,0.895143
8,hyundai,681,2376.967381,1649.454501,0.850023
7,vw,2121,3086.568078,2186.019099,0.845029
1,audi,1495,4289.597886,2825.460520,0.863752
3,bmw,1509,4324.461584,3041.687432,0.856351
2,mercedes,2382,6058.546019,3796.546605,0.704706


### SGD


In [60]:
# istanza del modello
SGD = SGDRegressor()
trainer = BrandModelTrainer(SGD)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)        # Performance VALID


Training models for 9 brands...

  ✓ mercedes done.
  ✓ toyota done.
  ✓ audi done.
  ✓ skoda done.
  ✓ opel done.
  ✓ vw done.
  ✓ bmw done.
  ✓ ford done.
  ✓ hyundai done.

Training Set Performance (Overall):
  RMSE: 3711.00
  MAE:  2200.54
  R²:   0.8544

Validation Set Performance (Overall):
  RMSE: 3612.77
  MAE:  2225.63
  R²:   0.8635

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7635 1613.678206 1115.091860 0.793605
  toyota  3774 2019.331256 1257.568266 0.894612
    ford 13114 2179.324272 1538.376797 0.793601
 hyundai  2723 2260.389983 1604.357461 0.853502
   skoda  3510 2482.110676 1578.267061 0.841988
      vw  8485 2996.414874 2094.878687 0.848804
     bmw  6037 4764.950747 3174.147112 0.826805
    audi  5973 4774.960143 3016.224185 0.832890
mercedes  9527 6099.778987 3720.886038 0.703415

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1909 1675.457420 1160.873481 0.784633
   skoda  876

,Brand,N,RMSE,MAE,R²
6,opel,1909,1675.457420,1160.873481,0.784633
0,skoda,876,2172.548364,1537.606103,0.878426
5,ford,3278,2192.821497,1554.287603,0.785677
4,toyota,944,2204.720848,1280.801133,0.898909
8,hyundai,681,2382.447667,1656.754084,0.849330
7,vw,2121,3125.296992,2195.014511,0.841115
1,audi,1495,4432.477740,2874.694753,0.854524
3,bmw,1509,4465.377653,3150.927449,0.846837
2,mercedes,2382,5953.918450,3826.639911,0.714817


### KNR

In [61]:
# istanza del modello
KNR = KNeighborsRegressor()
trainer = BrandModelTrainer(KNR)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

  ✓ mercedes done.
  ✓ toyota done.
  ✓ audi done.
  ✓ skoda done.
  ✓ opel done.
  ✓ vw done.
  ✓ bmw done.
  ✓ ford done.
  ✓ hyundai done.

Training Set Performance (Overall):
  RMSE: 2471.16
  MAE:  1424.75
  R²:   0.9355

Validation Set Performance (Overall):
  RMSE: 2916.47
  MAE:  1765.39
  R²:   0.9110

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7635 1120.548739  767.666562 0.900476
    ford 13114 1402.858334  931.752631 0.914475
  toyota  3774 1412.800663  866.810175 0.948413
 hyundai  2723 1558.521836 1034.436430 0.930355
   skoda  3510 1896.163132 1151.640513 0.907786
      vw  8485 2051.824622 1402.532092 0.929105
    audi  5973 3170.729673 2033.356337 0.926315
     bmw  6037 3615.665535 2197.962034 0.900277
mercedes  9527 3754.460985 2211.389566 0.887639

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1909 1412.418295  988.307491 0.846948
    ford 3278

,Brand,N,RMSE,MAE,R²
6,opel,1909,1412.418295,988.307491,0.846948
5,ford,3278,1743.493306,1167.524649,0.864511
4,toyota,944,1936.602529,1101.756568,0.922002
0,skoda,876,2018.034629,1414.439041,0.895104
8,hyundai,681,2116.531245,1372.547724,0.881087
7,vw,2121,2609.887144,1800.129844,0.889199
1,audi,1495,3625.923517,2386.992776,0.902650
3,bmw,1509,4129.083167,2671.563950,0.869038
2,mercedes,2382,4369.803245,2720.157347,0.846382


### Random Forest

In [62]:
# istanza del modello
RF = RandomForestRegressor()
trainer = BrandModelTrainer(RF)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...



KeyboardInterrupt: 

### Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_deep = MLPRegressor(
    hidden_layer_sizes=(512, 256, 128, 64),  
    activation='relu',
    solver='adam',
    alpha=0.0001,  
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=1000,
    batch_size=32,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,  # Più pazienza
    validation_fraction=0.15,
    verbose=True
)

trainer = BrandModelTrainer(mlp_deep)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 22043472.82945213
Validation score: 0.357357
Iteration 2, loss = 6860957.21686987
Validation score: 0.405066
Iteration 3, loss = 6575254.95262054
Validation score: 0.413837
Iteration 4, loss = 6442960.41884249
Validation score: 0.431650
Iteration 5, loss = 6376301.56310276
Validation score: 0.435515
Iteration 6, loss = 6232869.86717771
Validation score: 0.452481
Iteration 7, loss = 6105867.75164786
Validation score: 0.459479
Iteration 8, loss = 6000949.02302772
Validation score: 0.464047
Iteration 9, loss = 5878128.99980298
Validation score: 0.472874
Iteration 10, loss = 5793846.50678740
Validation score: 0.436217
Iteration 11, loss = 5699604.26011444
Validation score: 0.495438
Iteration 12, loss = 5646879.62600422
Validation score: 0.502912
Iteration 13, loss = 5550968.33816704
Validation score: 0.501755
Iteration 14, loss = 5492381.99910342
Validation score: 0.488008
Iteration 15, loss = 5499446.70140327
Validation score: 0.518157


,Brand,N,RMSE,MAE,R²
6,opel,1912,2241.580303,1472.888628,0.615157
1,skoda,914,2518.628206,1770.621197,0.817930
3,ford,3182,2711.415578,1905.984929,0.678478
7,hyundai,716,2834.978895,1875.300153,0.766054
8,toyota,939,3004.968333,1921.578237,0.767996
2,vw,2117,3552.523174,2254.535563,0.772561
4,audi,1527,5363.637946,3347.892006,0.758011
0,mercedes,2368,5466.803931,3363.666451,0.745159
5,bmw,1520,6703.984231,4663.633818,0.680939


In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_wide = MLPRegressor(
    hidden_layer_sizes=(1024,),  # Un solo layer MOLTO largo
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    learning_rate='adaptive',
    max_iter=1000,
    batch_size=64,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=30,
    validation_fraction=0.15,
    verbose=True
)

trainer = BrandModelTrainer(mlp_wide)

# fit
trainer.fit(X_train, y_train)

# performance overall
trainer.evaluate_train(X_train, y_train)
trainer.evaluate(X_val, y_val)

# performance per brand
trainer.evaluate_train_by_brand(X_train, y_train)
trainer.evaluate_by_brand(X_val, y_val)

Training models for 9 brands...

Iteration 1, loss = 89867252.42027950
Validation score: -7.095258
Iteration 2, loss = 84132912.70672533
Validation score: -6.277111
Iteration 3, loss = 72674494.71837711
Validation score: -5.025910
Iteration 4, loss = 57853238.06019688
Validation score: -3.606339
Iteration 5, loss = 42654326.40868299
Validation score: -2.276832
Iteration 6, loss = 29521201.43998935
Validation score: -1.220340
Iteration 7, loss = 19920235.85627457
Validation score: -0.518546
Iteration 8, loss = 14024697.01665131
Validation score: -0.127958
Iteration 9, loss = 11017590.03150420
Validation score: 0.053325
Iteration 10, loss = 9714728.71073372
Validation score: 0.126798
Iteration 11, loss = 9184441.24807563
Validation score: 0.158700
Iteration 12, loss = 8938955.33418435
Validation score: 0.177088
Iteration 13, loss = 8780513.20324910
Validation score: 0.191510
Iteration 14, loss = 8651905.80398754
Validation score: 0.204433
Iteration 15, loss = 8536816.31807630
Validation 

c:\Users\liber\anaconda3\envs\ML_env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 1, loss = 59720094.82837289
Validation score: -8.057110
Iteration 2, loss = 58426097.38610099
Validation score: -7.729972
Iteration 3, loss = 55440195.69688635
Validation score: -7.151093
Iteration 4, loss = 50972918.47013802
Validation score: -6.375271
Iteration 5, loss = 45424540.01031194
Validation score: -5.472425
Iteration 6, loss = 39257595.58113284
Validation score: -4.512306
Iteration 7, loss = 32968906.05869914
Validation score: -3.573501
Iteration 8, loss = 26967338.60440079
Validation score: -2.697114
Iteration 9, loss = 21572994.59291694
Validation score: -1.938556
Iteration 10, loss = 17003720.97395921
Validation score: -1.314438
Iteration 11, loss = 13334310.24512372
Validation score: -0.827652
Iteration 12, loss = 10566137.65220692
Validation score: -0.473100
Iteration 13, loss = 8607174.75095417
Validation score: -0.231696
Iteration 14, loss = 7297682.06690289
Validation score: -0.072901
Iteration 15, loss = 6452162.58494377
Validation score: 0.027376
Iteratio

c:\Users\liber\anaconda3\envs\ML_env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 2, loss = 97873698.51812124
Validation score: -3.894874
Iteration 3, loss = 97105746.90017499
Validation score: -3.842506
Iteration 4, loss = 95787015.43038695
Validation score: -3.762135
Iteration 5, loss = 93925014.03206630
Validation score: -3.655446
Iteration 6, loss = 91548253.67033295
Validation score: -3.524547
Iteration 7, loss = 88724563.22997545
Validation score: -3.372256
Iteration 8, loss = 85511248.92939383
Validation score: -3.201420
Iteration 9, loss = 81955918.27040510
Validation score: -3.017714
Iteration 10, loss = 78111701.37506971
Validation score: -2.817980
Iteration 11, loss = 74068317.34784649
Validation score: -2.610834
Iteration 12, loss = 69872678.08748966
Validation score: -2.396828
Iteration 13, loss = 65583971.71248122
Validation score: -2.181732
Iteration 14, loss = 61279753.13650904
Validation score: -1.965018
Iteration 15, loss = 56995178.30916391
Validation score: -1.750470
Iteration 16, loss = 52810998.50304195
Validation score: -1.543042
Ite

...

c:\Users\liber\anaconda3\envs\ML_env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Iteration 2, loss = 98324162.98309024
Validation score: -5.011775
Iteration 3, loss = 97972637.62936974
Validation score: -4.983153
Iteration 4, loss = 97363271.55921057
Validation score: -4.938300
Iteration 5, loss = 96477643.53299303
Validation score: -4.876617
Iteration 6, loss = 95307143.40183179
Validation score: -4.798195
Iteration 7, loss = 93866011.98302017
Validation score: -4.703533
Iteration 8, loss = 92159897.10906240
Validation score: -4.594594
Iteration 9, loss = 90222150.80634695
Validation score: -4.470986
Iteration 10, loss = 88054875.40617353
Validation score: -4.335377
Iteration 11, loss = 85694040.01866952
Validation score: -4.187456
Iteration 12, loss = 83146122.81623276
Validation score: -4.029667
Iteration 13, loss = 80435164.12648742
Validation score: -3.864233
Iteration 14, loss = 77604858.99609427
Validation score: -3.690748
Iteration 15, loss = 74670499.91052884
Validation score: -3.511079
Iteration 16, loss = 71650227.40577872
Validation score: -3.328589
Ite

c:\Users\liber\anaconda3\envs\ML_env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(



Training Set Performance (Overall):
  RMSE: 4485.39
  MAE:  2823.19
  R²:   0.7899

Validation Set Performance (Overall):
  RMSE: 4599.25
  MAE:  2926.26
  R²:   0.7678

Training Performance per Brand:
   Brand     N        RMSE         MAE       R²
    opel  7632 2126.243572 1438.529052 0.641379
  toyota  3779 2857.126889 1822.559498 0.801018
   skoda  3472 2887.890994 1943.114858 0.791618
 hyundai  2689 3049.805348 2035.654901 0.739752
    ford 13210 3066.919762 2221.973098 0.589381
      vw  8489 3631.962127 2412.343130 0.783283
    audi  5941 5889.799549 3744.032682 0.753213
     bmw  6025 6339.622845 4560.968455 0.687083
mercedes  9541 6626.760580 4396.565865 0.654738

Validation Performance per Brand:
   Brand    N        RMSE         MAE       R²
    opel 1912 2284.511289 1533.146243 0.600274
   skoda  914 2783.239814 1953.674246 0.777663
  toyota  939 3122.637043 1986.951622 0.749471
    ford 3182 3161.118676 2310.536992 0.562981
 hyundai  716 3182.329486 2086.973865 0.705215


,Brand,N,RMSE,MAE,R²
6,opel,1912,2284.511289,1533.146243,0.600274
1,skoda,914,2783.239814,1953.674246,0.777663
8,toyota,939,3122.637043,1986.951622,0.749471
3,ford,3182,3161.118676,2310.536992,0.562981
7,hyundai,716,3182.329486,2086.973865,0.705215
2,vw,2117,3637.085904,2463.986818,0.761604
4,audi,1527,6033.960971,3849.918413,0.693746
0,mercedes,2368,6659.535932,4533.473969,0.621827
5,bmw,1520,6665.818077,4740.085748,0.684562
